
# Machine Learning Assignment 2
## Classification of Dry Bean Varieties

### 1. Introduction and Problem Statement

Classification is a supervised machine learning task in which a model learns patterns from labelled data and uses them to predict the category of new observations.

In this assignment, the objective is to develop and evaluate five classification models using the Dry Bean dataset obtained from the UCI Machine Learning Repository.

The dataset contains measurements describing the shape and structure of dry beans. Using these measurements, the models will classify each bean into one of seven varieties.

The five classification algorithms considered are Logistic Regression, Decision Tree, K-Nearest Neighbors, Gaussian Naive Bayes and Support Vector Machine.

The performance of these models will be evaluated using Accuracy, AUC, Precision, Recall, F1-score and Matthews Correlation Coefficient (MCC). Their results will then be compared to identify the most suitable model for this classification problem.



### 2. Dataset Selection and Description

The Dry Bean dataset is a publicly available classification dataset obtained from the UCI Machine Learning Repository. It contains measurements extracted from images of dry bean grains using a computer vision system.

The dataset consists of 13,611 observations and 16 numerical input features. The target variable, Class, represents one of seven dry bean varieties: SEKER, BARBUNYA, BOMBAY, CALI, DERMASON, HOROZ and SIRA.

The input features describe different physical characteristics of the beans, including area, perimeter, axis lengths, eccentricity, roundness and shape factors.

The objective is to use these numerical measurements to predict the corresponding bean variety.

**Dataset source:** https://archive.ics.uci.edu/dataset/602/dry


### 3. Environment Setup and Verification

Before starting the implementation, the Python environment is verified to ensure that the required libraries are installed and working correctly. This helps avoid dependency-related issues during dataset processing and model development.

In [3]:
import sys
import numpy as np
import pandas as pd
import sklearn

print("Python version:", sys.version.split()[0])
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("Environment verification completed successfully!")

Python version: 3.13.17
NumPy version: 2.5.4
Pandas version: 3.1.1
Scikit-learn version: 1.9.1
Environment verification completed successfully!



### 4. Importing Required Libraries

The following libraries are used for data handling, visualization, preprocessing, model development and performance evaluation.

Pandas and NumPy are used for data manipulation and numerical operations. Matplotlib and Seaborn are used for data visualization. Scikit-learn provides the preprocessing techniques, classification algorithms and evaluation metrics required for this assignment.


In [2]:

# Data handling
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset retrieval
from ucimlrepo import fetch_ucirepo

# Data preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

# Model saving
import joblib

print("All required libraries imported successfully!")


All required libraries imported successfully!



### 5. Loading the Dataset

The Dry Bean dataset is obtained from the UCI Machine Learning Repository using the `ucimlrepo` library.

The dataset is separated into two parts: input features and the target variable. The input features represent the physical measurements of the dry beans, while the target variable represents their corresponding bean varieties.

Both are combined into a single DataFrame to make the dataset easier to inspect and process in the subsequent stages.


In [4]:

# Fetch the Dry Bean dataset from UCI
dry_bean = fetch_ucirepo(id=602)

# Extract input features and target variable
X = dry_bean.data.features
y = dry_bean.data.targets

# Combine features and target into a single DataFrame
df = pd.concat([X, y], axis=1)

# Display dataset information
print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nFirst five records:")
display(df.head())


Dataset loaded successfully!
Dataset shape: (13611, 17)

First five records:


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRatio,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,Roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272751,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER



### 6. Initial Data Inspection

After loading the dataset, an initial inspection is performed to understand its structure and quality before applying any preprocessing techniques.

This step helps identify the number of observations and features, data types, missing values and duplicate records. It also provides an overview of the target variable and its class distribution.

The findings from this inspection will help determine whether any data cleaning is required before model development.


#### 6.1 Dataset Dimensions

The dataset dimensions indicate the total number of observations and variables available for analysis. This helps verify whether the dataset satisfies the minimum size requirements specified in the assignment.

In [12]:
print("Number of observations:", df.shape[0])
print("Number of columns:", df.shape[1])
print("Dataset dimensions:", df.shape)

Number of observations: 13611
Number of columns: 17
Dataset dimensions: (13611, 17)


#### 6.2 Column Names and Data Types

The column names identify the variables available in the dataset, while their data types indicate how each variable is represented. This helps determine whether the features are suitable for numerical analysis and classification.

In [14]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Column names:
['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRatio', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'Roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4', 'Class']

Data types:
Area                 int64
Perimeter          float64
MajorAxisLength    float64
MinorAxisLength    float64
AspectRatio        float64
Eccentricity       float64
ConvexArea           int64
EquivDiameter      float64
Extent             float64
Solidity           float64
Roundness          float64
Compactness        float64
ShapeFactor1       float64
ShapeFactor2       float64
ShapeFactor3       float64
ShapeFactor4       float64
Class                  str
dtype: object


#### 6.3 Dataset Information

The dataset information provides a summary of the columns, their data types and the number of non-null observations. This helps understand the overall structure of the dataset and identify potential data quality issues.

In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13611 entries, 0 to 13610
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Area             13611 non-null  int64  
 1   Perimeter        13611 non-null  float64
 2   MajorAxisLength  13611 non-null  float64
 3   MinorAxisLength  13611 non-null  float64
 4   AspectRatio      13611 non-null  float64
 5   Eccentricity     13611 non-null  float64
 6   ConvexArea       13611 non-null  int64  
 7   EquivDiameter    13611 non-null  float64
 8   Extent           13611 non-null  float64
 9   Solidity         13611 non-null  float64
 10  Roundness        13611 non-null  float64
 11  Compactness      13611 non-null  float64
 12  ShapeFactor1     13611 non-null  float64
 13  ShapeFactor2     13611 non-null  float64
 14  ShapeFactor3     13611 non-null  float64
 15  ShapeFactor4     13611 non-null  float64
 16  Class            13611 non-null  str    
dtypes: float64(14), int64(2

#### 6.4 Descriptive Statistical Summary

Descriptive statistics provide an overview of the numerical features through measures such as mean, standard deviation, minimum, maximum and quartiles. These values help understand the distribution and scale of the features.

In [16]:
display(df.describe().T)

,count,mean,std,min,25%,50%,75%,max
Area,13611.0,53048.284549,29324.095717,20420.000000,36328.000000,44652.000000,61332.000000,254616.000000
Perimeter,13611.0,855.283459,214.289696,524.736000,703.523500,794.941000,977.213000,1985.370000
MajorAxisLength,13611.0,320.141867,85.694186,183.601165,253.303633,296.883367,376.495012,738.860154
MinorAxisLength,13611.0,202.270714,44.970091,122.512653,175.848170,192.431733,217.031741,460.198497
AspectRatio,13611.0,1.583242,0.246678,1.024868,1.432307,1.551124,1.707109,2.430306
Eccentricity,13611.0,0.750895,0.092002,0.218951,0.715928,0.764441,0.810466,0.911423
ConvexArea,13611.0,53768.200206,29774.915817,20684.000000,36714.500000,45178.000000,62294.000000,263261.000000
EquivDiameter,13611.0,253.064220,59.177120,161.243764,215.068003,238.438026,279.446467,569.374358
Extent,13611.0,0.749733,0.049086,0.555315,0.718634,0.759859,0.786851,0.866195
Solidity,13611.0,0.987143,0.004660,0.919246,0.985670,0.988283,0.990013,0.994677


#### 6.5 Missing Value Analysis

Missing values can affect the performance of machine learning algorithms. Therefore, the number of missing values in each column is examined to determine whether any imputation or other handling technique is required.

In [17]:
print("Missing values in each column:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

Missing values in each column:
Area               0
Perimeter          0
MajorAxisLength    0
MinorAxisLength    0
AspectRatio        0
Eccentricity       0
ConvexArea         0
EquivDiameter      0
Extent             0
Solidity           0
Roundness          0
Compactness        0
ShapeFactor1       0
ShapeFactor2       0
ShapeFactor3       0
ShapeFactor4       0
Class              0
dtype: int64

Total missing values: 0


#### 6.6 Duplicate Record Analysis

Duplicate records represent repeated observations in the dataset. Identifying them helps assess data quality and determine whether duplicate removal is necessary before model development.

In [18]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate records:", duplicate_count)

Number of duplicate records: 68


#### 6.7 Target Variable Distribution

The target variable represents the dry bean variety to be predicted. Examining its class distribution helps understand the number of observations belonging to each variety and identify any potential class imbalance.

In [19]:
print("Number of unique classes:", df["Class"].nunique())

print("\nClass distribution:")
print(df["Class"].value_counts())

Number of unique classes: 7

Class distribution:
Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64


### 6.8 Summary of Initial Data Inspection

The initial inspection confirms that the dataset contains 13,611 observations, 16 numerical input features and one target variable named `Class`. All input features have numerical data types, and no missing values were identified.

A total of 68 duplicate records were found. These records will be removed before model development to avoid repeated observations. The target variable contains seven classes, with some variation in the number of observations across classes.

Since no missing values are present, no imputation is required. The dataset is suitable for further preprocessing and multi-class classification.

### 7.1 Removing Duplicate Records

The initial inspection identified 68 duplicate records in the dataset. These duplicate observations are removed to retain only unique records.

After removing duplicates, the dataset index is reset to maintain a continuous sequence. The updated dataset dimensions and duplicate count are checked to verify the cleaning process.

In [20]:
# Remove duplicate records and reset the index
df = df.drop_duplicates().reset_index(drop=True)

# Update input features and target variable
X = df.drop(columns=["Class"])
y = df["Class"]

# Verify the cleaning process
print("Dataset shape after removing duplicates:", df.shape)
print("Remaining duplicate records:", df.duplicated().sum())
print("Number of input features:", X.shape[1])
print("Number of target classes:", y.nunique())

Dataset shape after removing duplicates: (13543, 17)
Remaining duplicate records: 0
Number of input features: 16
Number of target classes: 7
